# European Bike Sharing Analytics

**Comparative analysis of observed demand, trip behavior, station activity, and fleet utilization in Berlin, Warszawa, Dresden, and Bilbao.**

The central question is: *How do bike-sharing usage patterns differ across selected European cities?* The analysis uses normalized metrics when observation coverage differs and avoids causal claims that the dataset cannot support.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import CITY_COLORS

PROCESSED = ROOT / "data" / "processed"
def load(name):
    return pd.read_parquet(PROCESSED / f"{name}.parquet")

kpis = load("city_kpis")
profile = load("city_profile")
quality = load("data_quality")
hourly = load("hourly_demand")
weekday = load("weekday_weekend")
characteristics = load("trip_characteristics")
stations = load("station_performance")
routes = load("popular_routes")
fleet = load("bike_utilization")
bike_types = load("bike_type_analysis")
metadata = json.loads((PROCESSED / "analysis_metadata.json").read_text())
pd.options.display.float_format = "{:,.3f}".format
print(f"Loaded {metadata['analytical_rows']:,} prepared trips across {metadata['cities']} cities.")

Loaded 6,839,701 prepared trips across 4 cities.


## 1. Project objective

Compare cities on demand, time patterns, trip characteristics, stations, routes, and observed-bike utilization using transparent descriptive analytics.

## 2. Dataset overview

The source is the [TUM European Bike Sharing Dataset](https://github.com/TUMFTM/european-bike-sharing-dataset), published with a 2025 *Transportation* article and licensed under CC BY-NC 4.0. The publication year is distinct from the observed trip period. Source timestamps are Unix epoch seconds in UTC and are converted to local time with the upstream city timezone.

In [2]:
overview = pd.DataFrame({
    "Prepared trips": [metadata["analytical_rows"]],
    "Observation start": [metadata["observation_start"]],
    "Observation end": [metadata["observation_end"]],
    "Cities": [metadata["cities"]],
    "Local trip subset (MB)": [metadata["interim_parquet_bytes"] / 1_000_000],
})
display(overview.style.format({"Prepared trips": "{:,.0f}", "Local trip subset (MB)": "{:.1f}"}))

,Prepared trips,Observation start,Observation end,Cities,Local trip subset (MB)
0,"6,839,701",2022-08-26,2023-07-15,4,226.0


## 3. City selection

All preferred cities passed the explicit minimums of 1,000 trips, 14 active days, and 80% valid duration coverage. Warszawa combines the upstream IDs `210` (`Warszawa`) and `812` (`Warszawa - 2023`), which represent the same city in separate source records. No replacement city was needed.

In [3]:
selection_columns = ["city", "source_city_ids", "total_trips", "observation_start", "observation_end", "active_days", "stations", "unique_bikes", "valid_duration_share", "valid_distance_share"]
display(profile.loc[profile["city"].isin(kpis["city"]), selection_columns]
        .sort_values("total_trips", ascending=False)
        .style.format({"total_trips": "{:,.0f}", "valid_duration_share": "{:.2%}", "valid_distance_share": "{:.2%}"}))

,city,source_city_ids,total_trips,observation_start,observation_end,active_days,stations,unique_bikes,valid_duration_share,valid_distance_share
0,Warszawa,"210, 812","2,784,538",2022-08-26 00:00:00,2023-07-15 00:00:00,238,330,9125,99.90%,100.00%
2,Bilbao,532,"1,438,586",2022-08-29 00:00:00,2023-07-15 00:00:00,309,44,717,99.95%,100.00%
3,Dresden,685,"1,403,079",2022-08-26 00:00:00,2023-07-15 00:00:00,312,278,1503,99.97%,100.00%
4,Berlin,362,"1,213,498",2022-08-26 00:00:00,2023-07-15 00:00:00,312,2815,7652,99.90%,100.00%


## 4. Data quality

In [4]:
quality_view = quality.copy()
for field in ["invalid_timestamp_rows", "invalid_duration_rows", "invalid_distance_rows", "missing_start_station_rows", "missing_end_station_rows"]:
    quality_view[field.replace("_rows", "_share")] = quality_view[field] / quality_view["source_rows"]
display(quality_view[["city", "source_rows", "duplicate_rows", "invalid_timestamp_share", "invalid_duration_share", "invalid_distance_share", "missing_start_station_share", "missing_end_station_share", "retained_rows"]]
        .style.format({"source_rows": "{:,.0f}", "retained_rows": "{:,.0f}", "invalid_timestamp_share": "{:.2%}", "invalid_duration_share": "{:.2%}", "invalid_distance_share": "{:.2%}", "missing_start_station_share": "{:.1%}", "missing_end_station_share": "{:.1%}"}))

,city,source_rows,duplicate_rows,invalid_timestamp_share,invalid_duration_share,invalid_distance_share,missing_start_station_share,missing_end_station_share,retained_rows
0,Berlin,"1,213,498",0,0.00%,0.10%,0.00%,28.5%,35.5%,"1,213,498"
1,Bilbao,"1,438,586",0,0.00%,0.05%,0.00%,0.0%,0.0%,"1,438,586"
2,Dresden,"1,403,079",0,0.00%,0.03%,0.00%,73.2%,74.8%,"1,403,079"
3,Warszawa,"2,784,538",0,0.00%,0.10%,0.00%,3.7%,4.8%,"2,784,538"


No exact duplicate rows or invalid timestamps were found in the selected cities. Duration outside 1 minute to 6 hours is treated as unavailable for duration statistics, while distance is treated as unavailable when non-positive, above 100 km, or associated with an implied speed above 60 km/h. Such characteristic-level issues do not remove otherwise valid trips from demand counts. Missing station IDs are retained because Berlin and Dresden include substantial free-floating activity.

## 5. Core KPIs

In [5]:
kpi_view = kpis[["city", "total_trips", "active_days", "trips_per_active_day", "stations", "unique_bikes", "median_duration_min", "median_distance_km", "weekend_share", "peak_hour", "fleet_utilization_proxy"]]
display(kpi_view.style.format({"total_trips": "{:,.0f}", "trips_per_active_day": "{:,.0f}", "median_duration_min": "{:.1f}", "median_distance_km": "{:.2f}", "weekend_share": "{:.1%}", "fleet_utilization_proxy": "{:.2f}"}))
fig = px.bar(kpis.sort_values("trips_per_active_day"), x="trips_per_active_day", y="city", orientation="h", color="city", color_discrete_map=CITY_COLORS, title="Trips per Active Day", labels={"trips_per_active_day": "Trips per active day", "city": "City"})
fig.update_layout(showlegend=False)
fig.show()

,city,total_trips,active_days,trips_per_active_day,stations,unique_bikes,median_duration_min,median_distance_km,weekend_share,peak_hour,fleet_utilization_proxy
0,Warszawa,"2,784,538",238,"11,700",330,9125,15.0,1.74,27.8%,16,1.28
1,Bilbao,"1,438,586",309,"4,656",44,717,12.0,1.37,16.7%,14,6.49
2,Dresden,"1,403,079",312,"4,497",278,1503,9.0,1.18,23.0%,17,2.99
3,Berlin,"1,213,498",312,"3,889",2815,7652,15.0,1.85,28.7%,18,0.51


## 6. Demand by hour

In [6]:
fig = px.line(hourly, x="hour", y="trip_share", color="city", markers=True, color_discrete_map=CITY_COLORS, title="Hourly Demand Profile", labels={"hour": "Local hour", "trip_share": "Share of city trips"})
fig.update_yaxes(tickformat=".1%")
fig.show()

## 7. Weekday vs weekend

In [7]:
fig = px.line(weekday.sort_values(["city", "day_of_week"]), x="day_name", y="trips_per_active_day", color="city", markers=True, line_dash="day_type", color_discrete_map=CITY_COLORS, title="Trips per Observed Day of Week", labels={"day_name": "Day", "trips_per_active_day": "Trips per observed day"})
fig.show()
day_type = weekday.groupby(["city", "day_type"], as_index=False).agg({"trips": "sum", "active_days": "sum"})
day_type["trips_per_active_day"] = day_type["trips"] / day_type["active_days"]
display(day_type.style.format({"trips": "{:,.0f}", "trips_per_active_day": "{:,.0f}"}))

,city,day_type,trips,active_days,trips_per_active_day
0,Berlin,Weekday,"865,696",224,"3,865"
1,Berlin,Weekend,"347,802",88,"3,952"
2,Bilbao,Weekday,"1,198,870",223,"5,376"
3,Bilbao,Weekend,"239,716",86,"2,787"
4,Dresden,Weekday,"1,080,722",224,"4,825"
5,Dresden,Weekend,"322,357",88,"3,663"
6,Warszawa,Weekday,"2,009,680",173,"11,617"
7,Warszawa,Weekend,"774,858",65,"11,921"


## 8. Trip duration and distance

In [8]:
trip_metrics = characteristics.melt(id_vars="city", value_vars=["median_duration_min", "median_distance_km"], var_name="metric", value_name="value")
trip_metrics["metric"] = trip_metrics["metric"].map({"median_duration_min": "Median duration (min)", "median_distance_km": "Median distance (km)"})
fig = px.bar(trip_metrics, x="city", y="value", color="city", facet_col="metric", color_discrete_map=CITY_COLORS, title="Median Trip Characteristics")
fig.update_yaxes(matches=None)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 9. Station and route analysis

In [9]:
station_plot = stations.loc[stations["city_rank"] <= 5].copy()
fig = px.bar(station_plot.sort_values(["city", "total_activity"]), x="total_activity", y="station_name", color="city", facet_col="city", facet_col_wrap=2, orientation="h", color_discrete_map=CITY_COLORS, title="Top Five Stations by Starts and Ends", labels={"total_activity": "Starts + ends", "station_name": "Station"})
fig.update_layout(showlegend=False, height=750)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()
display(routes.loc[routes["city_rank"] <= 3, ["city", "city_rank", "start_station_name", "end_station_name", "trips"]]
        .style.format({"trips": "{:,.0f}"}))

,city,city_rank,start_station_name,end_station_name,trips
0,Berlin,1,Unknown station,Unknown station,876
1,Berlin,2,Unknown station,Unknown station,814
2,Berlin,3,Unknown station,Unknown station,679
10,Bilbao,1,02-BLAS OTERO,21-SARRIKO,"10,357"
11,Bilbao,2,21-SARRIKO,02-BLAS OTERO,"9,055"
12,Bilbao,3,01-LEVANTE,21-SARRIKO,"8,950"
20,Dresden,1,MOBIpunkt Bahnhof Neustadt,MOBIpunkt Louisenstraße,"1,040"
21,Dresden,2,MOBIpunkt TU Dresden,SLUB Zentralbibliothek,"1,004"
22,Dresden,3,MOBIpunkt Louisenstraße,MOBIpunkt Bahnhof Neustadt,867
30,Warszawa,1,Metro Ursynów I,Jastrzębowskiego - SGGW,"3,802"


## 10. Fleet and bike-type analysis

In [10]:
fig = px.bar(fleet.sort_values("fleet_utilization_proxy"), x="fleet_utilization_proxy", y="city", orientation="h", color="city", color_discrete_map=CITY_COLORS, title="Fleet Utilization Proxy", labels={"fleet_utilization_proxy": "Trips per observed bike per active day", "city": "City"})
fig.update_layout(showlegend=False)
fig.show()
fig = px.bar(bike_types, x="city", y="trip_share", color="propulsion_type", title="Trips by Propulsion Type", labels={"trip_share": "Share of city trips", "propulsion_type": "Propulsion type"}, color_discrete_map={"electric_assist": "#38BDF8", "human": "#334155", "unknown": "#CBD5E1"})
fig.update_yaxes(tickformat=".0%")
fig.show()

## 11. Cross-city comparison

In [11]:
fig = px.scatter(kpis, x="trips_per_active_day", y="fleet_utilization_proxy", size="total_trips", color="city", text="city", color_discrete_map=CITY_COLORS, title="Observed Demand vs Fleet Utilization Proxy", labels={"trips_per_active_day": "Trips per active day", "fleet_utilization_proxy": "Trips per observed bike per active day"})
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
fig.show()

## 12. Key findings

1. **Warszawa recorded the highest demand:** 2.78 million trips and about 11,700 trips per active day, despite having fewer observed days than the other cities.
2. **Absolute demand and observed-bike utilization tell different stories:** Bilbao had 4,656 trips per active day but the highest fleet utilization proxy at 6.49 trips per observed bike per active day; Berlin's proxy was 0.51.
3. **Peak timing differed by city:** Bilbao peaked at 14:00, Warszawa at 16:00, Dresden at 17:00, and Berlin at 18:00 in local time.
4. **Weekend patterns were not uniform:** observed weekend demand per day was slightly higher in Berlin and Warszawa, but materially lower in Dresden and Bilbao.
5. **Trips were shortest in Dresden:** the median was 9 minutes and 1.18 km. Berlin had the highest median distance at 1.85 km; Berlin and Warszawa both had 15-minute median durations.
6. **Station coverage varied sharply:** almost every Bilbao trip had station IDs, while roughly three quarters of Dresden starts and ends were missing, consistent with substantial free-floating activity. Station rankings should therefore be interpreted within each city.
7. **Observed propulsion mix differed:** Bilbao trips were entirely mapped to electric-assist bikes, Warszawa had an observed electric-assist share of 11.5%, and Berlin and Dresden were almost entirely human-powered in the source mapping.

## 13. Practical insights

- Schedule rebalancing attention around each city's observed peak instead of applying one common peak-hour assumption.
- Use normalized demand alongside totals when planning across cities with unequal coverage.
- Review Bilbao's high trips-per-observed-bike value as a potential high-intensity operating pattern, while treating it as a proxy rather than verified capacity utilization.
- Use station-level actions primarily in Bilbao and Warszawa; free-floating coverage limits station conclusions in Berlin and especially Dresden.
- Keep propulsion-specific monitoring most prominent in Bilbao and Warszawa, where electric-assist bikes account for a meaningful observed share.

## 14. Limitations

- Observation coverage is unequal: Warszawa has 238 active days, versus 309–312 for the other cities.
- `bike_id` counts bikes observed in trips and may not represent complete registered fleet capacity; utilization is explicitly a proxy.
- Station analysis excludes free-floating trips and is less representative in Berlin and Dresden.
- Distance is straight-line endpoint distance from the upstream dataset, not routed cycling distance.
- The dataset supports descriptive associations, not causal explanations involving weather, infrastructure, pricing, or demographics.
- Warszawa combines two official city records. Results are city-level and do not claim that the underlying operating configurations were identical.